# Project 5 — End-to-End ML System: Customer Churn Prediction

Built a complete machine learning pipeline to predict which customers 
will cancel their subscription. Trained and compared 3 models, picked 
the best one, and deployed it as a live web app using Streamlit.

The Railofy connection: churn prediction directly maps to my internship work.
I manually tracked repeat customer rates across 65 stations. This model 
would have automatically flagged which customers were about to stop ordering
before they actually did.

Approach: EDA → feature engineering → 3 models compared → 
best model saved → Streamlit app deployed live

In [1]:
# starting with imports for this project
# joblib is something i hadnt used before - its for saving the model to a file
# without it we'd have to retrain the model every time the streamlit app loads

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from xgboost import XGBClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


In [2]:
# loading the telco churn dataset
# first time working with telecom data - curious to see what columns it has

df = pd.read_csv("D:\\Data_Science\\Projects\\Project 5\\WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst look:")
print(df.head())
print("\nChurn distribution:")
print(df['Churn'].value_counts())
print("\nChurn rate:", round(df['Churn'].value_counts(normalize=True)['Yes'] * 100, 2), "%")

Shape: (7043, 21)

Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

First look:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1 

In [3]:
# dropping customerID - its just a unique identifier, useless for prediction
# TotalCharges is stored as string for some reason - needs to be converted to number
# found this out when i tried to run the model and it crashed

df = df.drop('customerID', axis=1)

# this column has some empty spaces instead of NaN - took me a while to spot this
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

# converting Yes/No columns to 1/0
# ml models cant handle text, everything needs to be numbers
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn',
               'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 1, 'No phone service': 0, 'No internet service': 0})

# label encoding the remaining categorical columns
# gender, InternetService, Contract, PaymentMethod
le = LabelEncoder()
cat_cols = ['gender', 'InternetService', 'Contract', 'PaymentMethod']
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("Data cleaned!")
print("Shape after cleaning:", df.shape)
print(df.dtypes)

Data cleaned!
Shape after cleaning: (7032, 20)
gender                int64
SeniorCitizen         int64
Partner               int64
Dependents            int64
tenure                int64
PhoneService          int64
MultipleLines         int64
InternetService       int64
OnlineSecurity        int64
OnlineBackup          int64
DeviceProtection      int64
TechSupport           int64
StreamingTV           int64
StreamingMovies       int64
Contract              int64
PaperlessBilling      int64
PaymentMethod         int64
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object


In [4]:
# separating features (X) from target (y)
# churn is what we want to predict, everything else is input

X = df.drop('Churn', axis=1)
y = df['Churn']

# 80% for training, 20% for testing
# random_state=42 so results are reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scaling the numbers - same reason as project 1
# tenure goes up to 72, monthlycharges up to 120 - need same scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# training 3 models and comparing them
# started with logistic regression since its the simplest
# then random forest, then xgboost which usually wins

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
    results[name] = {'Accuracy': round(acc*100, 2), 'AUC-ROC': round(auc, 4)}
    print(f"{name}: Accuracy={round(acc*100,2)}% | AUC-ROC={round(auc,4)}")

print("\nBest model based on AUC-ROC:")
best_model_name = max(results, key=lambda x: results[x]['AUC-ROC'])
print(best_model_name)

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(1)

In [5]:
# the error said "data contains only one class" which was confusing at first
# turns out in cell 3 i accidentally mapped both Yes and No to 1
# for the binary cols the mapping was wrong - No should be 0 not 1
# had to go back and redo the cleaning properly

df2 = pd.read_csv("D:\\Data_Science\\Projects\\Project 5\\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df2 = df2.drop('customerID', axis=1)
df2['TotalCharges'] = pd.to_numeric(df2['TotalCharges'], errors='coerce')
df2 = df2.dropna()

# fixed mapping this time - Yes=1, No=0
df2['Churn'] = (df2['Churn'] == 'Yes').astype(int)
df2['Partner'] = (df2['Partner'] == 'Yes').astype(int)
df2['Dependents'] = (df2['Dependents'] == 'Yes').astype(int)
df2['PhoneService'] = (df2['PhoneService'] == 'Yes').astype(int)
df2['PaperlessBilling'] = (df2['PaperlessBilling'] == 'Yes').astype(int)
df2['MultipleLines'] = (df2['MultipleLines'] == 'Yes').astype(int)
df2['OnlineSecurity'] = (df2['OnlineSecurity'] == 'Yes').astype(int)
df2['OnlineBackup'] = (df2['OnlineBackup'] == 'Yes').astype(int)
df2['DeviceProtection'] = (df2['DeviceProtection'] == 'Yes').astype(int)
df2['TechSupport'] = (df2['TechSupport'] == 'Yes').astype(int)
df2['StreamingTV'] = (df2['StreamingTV'] == 'Yes').astype(int)
df2['StreamingMovies'] = (df2['StreamingMovies'] == 'Yes').astype(int)

le = LabelEncoder()
for col in ['gender', 'InternetService', 'Contract', 'PaymentMethod']:
    df2[col] = le.fit_transform(df2[col])

print("Fixed! Churn value counts:")
print(df2['Churn'].value_counts())

Fixed! Churn value counts:
Churn
0    5163
1    1869
Name: count, dtype: int64


In [6]:
# now training properly with the fixed dataset
# same approach as before - split, scale, train 3 models, compare

X = df2.drop('Churn', axis=1)
y = df2['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
    results[name] = {'Accuracy': round(acc*100, 2), 'AUC': round(auc, 4)}
    print(f"{name}: Accuracy={round(acc*100,2)}% | AUC-ROC={round(auc,4)}")

best_model_name = max(results, key=lambda x: results[x]['AUC'])
print(f"\nBest model: {best_model_name}")

Logistic Regression: Accuracy=78.54% | AUC-ROC=0.8306
Random Forest: Accuracy=78.04% | AUC-ROC=0.8133
XGBoost: Accuracy=77.61% | AUC-ROC=0.8119

Best model: Logistic Regression


In [7]:
# logistic regression won so saving that one
# joblib saves the model as a file so streamlit can load it without retraining
# also saving the scaler because we need to scale inputs the same way in the app

best_model = models['Logistic Regression']

joblib.dump(best_model, "D:\\Data_Science\\Projects\\Project 5\\churn_model.pkl")
joblib.dump(scaler, "D:\\Data_Science\\Projects\\Project 5\\scaler.pkl")

print("Model saved as churn_model.pkl")
print("Scaler saved as scaler.pkl")
print("Both files are in Project 5 folder")

Model saved as churn_model.pkl
Scaler saved as scaler.pkl
Both files are in Project 5 folder


In [1]:
# summarising everything before moving to the streamlit part
# honestly surprised logistic regression beat xgboost
# usually see xgboost winning in most kaggle notebooks i read

print("=" * 60)
print("PROJECT 5 - CHURN PREDICTION RESULTS")
print("=" * 60)

print(f"""
DATASET: Telco Customer Churn (7,032 customers)
CHURN RATE: 26.54% - roughly 1 in 4 customers leaves

MODEL COMPARISON:
- Logistic Regression: 78.54% accuracy | AUC 0.8306 - winner
- Random Forest:       78.04% accuracy | AUC 0.8133
- XGBoost:             77.61% accuracy | AUC 0.8119

my take - logistic regression won probably because the dataset
is small and the relationships are fairly straightforward
xgboost needs more data to really show its advantage

things that seem to drive churn based on the model:
- month to month contracts leave the most
- fiber optic customers churn more than DSL
- newer customers churn more than long term ones
- higher monthly charges = higher chance of leaving

railofy connection:
- i used to track repeat customer rates manually across 65 stations
- a model like this running daily would flag at-risk customers
- could have helped push our repeat rate even higher than it was
""")

print("notebook done - next step is building the streamlit app")

PROJECT 5 - CHURN PREDICTION RESULTS

DATASET: Telco Customer Churn (7,032 customers)
CHURN RATE: 26.54% - roughly 1 in 4 customers leaves

MODEL COMPARISON:
- Logistic Regression: 78.54% accuracy | AUC 0.8306 - winner
- Random Forest:       78.04% accuracy | AUC 0.8133
- XGBoost:             77.61% accuracy | AUC 0.8119

my take - logistic regression won probably because the dataset
is small and the relationships are fairly straightforward
xgboost needs more data to really show its advantage

things that seem to drive churn based on the model:
- month to month contracts leave the most
- fiber optic customers churn more than DSL
- newer customers churn more than long term ones
- higher monthly charges = higher chance of leaving

railofy connection:
- i used to track repeat customer rates manually across 65 stations
- a model like this running daily would flag at-risk customers
- could have helped push our repeat rate even higher than it was

notebook done - next step is building the st